# Chatbot IA Poulina – RAG + Random Forest

Recommandation de souches & laboratoires

---



| Version | 2.0 – Avril 2025 |
|------------|------------------|
| Modèle Embedding | TF-IDF sparse (rapide, local) |
| Modèle Prédiction | Random Forest Classifier |
| Modèle Génération | Mistral Large (API) |
| Vector Store | ChromaDB (persistant) |

## Table des matières

| # | Section | Description |
|---|---------|-------------|
| 1 | [Installation](#1) | Dépendances Python |
| 2 | [Configuration & Imports](#2) | Modèles, constantes, LLM |
| 3 | [Chargement des données](#3) | CSV Analyses + Laboratoires |
| 4 | [Nettoyage](#4) | Normalisation, booléens, valeurs manquantes |
| 5 | [Chunking](#5) | Transformation lignes → texte structuré |
| **6** | [**Embeddings TF-IDF rapides**](#6) | **Vectorisation sparse avec cache disque** |
| **7** | [**Modèle Random Forest**](#7) | **Prédiction meilleure souche + labo** |
| 8 | [Indexation ChromaDB](#8) | 2 collections vectorielles |
| 9 | [Retrieval & Routage](#9) | Sélection intelligente de collection |
| 10 | [Pipeline RAG complet](#10) | Retrieval → RF → Mistral → Réponse |
| 11 | [Tests métier](#11) | 6 scénarios réels Poulina |

<a id='1'></a>
## 1. Installation des dépendances

In [2]:
%pip install -r requirements.txt -qU
#chromadb langchain_mistralai python-dotenv scikit-learn joblib tqdm -qU

Note: you may need to restart the kernel to use updated packages.


<a id='2'></a>
## 2. Configuration & Imports


In [3]:
import os, json, uuid, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

# ChromaDB & Mistral
import chromadb
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage, SystemMessage

warnings.filterwarnings("ignore")
load_dotenv(".env")

# ── Fichiers source ────────────────────────────────────────────────────────
CSV_ANALYSES = "poulina_dataset_5000.csv"
CSV_LABOS    = "poulina_laboratoires_5000.csv"

# ── ChromaDB ───────────────────────────────────────────────────────────────
COLLECTION_ANALYSES = "poulina_analyses"
COLLECTION_LABOS    = "poulina_laboratoires"
CHROMA_PATH         = "./chroma_poulina"

# ── RAG ────────────────────────────────────────────────────────────────────
TOP_K        = 5
SCORE_THRESH = 0.10   # seuil cosine TF-IDF (valeurs plus basses qu'avec e5)

# ── Modèles sauvegardés ────────────────────────────────────────────────────
RF_SOUCHE_PATH  = "./models/rf_souche.pkl"
RF_LABO_PATH    = "./models/rf_labo.pkl"
TFIDF_A_PATH    = "./models/tfidf_analyses.pkl"
TFIDF_L_PATH    = "./models/tfidf_labos.pkl"
Path("./models").mkdir(exist_ok=True)

# ── LLM Mistral ───────────────────────────────────────────────────────────
MISTRAL_MODEL = "mistral-large-latest"
llm = ChatMistralAI(model=MISTRAL_MODEL, temperature=0.0)

print(" Configuration OK")
print(f"   LLM          : {MISTRAL_MODEL}")
print(f"   Embeddings   : TF-IDF sparse (sklearn) – pas de GPU requis")
print(f"   Prédiction   : Random Forest Classifier")

 Configuration OK
   LLM          : mistral-large-latest
   Embeddings   : TF-IDF sparse (sklearn) – pas de GPU requis
   Prédiction   : Random Forest Classifier


<a id='3'></a>
## 3. Chargement des données

In [ ]:
def load_csv(path: str, label: str) -> pd.DataFrame | None:
    if not Path(path).exists():
        print(f"Fichier introuvable : {path}")
        return None
    df = pd.read_csv(path)
    print(f" {label} : {len(df):,} lignes × {len(df.columns)} colonnes")
    return df

df_analyses = load_csv(CSV_ANALYSES, "Analyses")
df_labos    = load_csv(CSV_LABOS,    "Laboratoires")

if df_analyses is not None:
    print("\nColonnes analyses :", list(df_analyses.columns))
if df_labos is not None:
    print("\nColonnes labos    :", list(df_labos.columns))

 Analyses : 5,000 lignes × 20 colonnes
 Laboratoires : 500 lignes × 41 colonnes

Colonnes analyses : ['id_centre', 'ville', 'region', 'type_production', 'capacite', 'temperature_moyenne', 'humidite', 'altitude', 'biosecurite_score', 'historique_maladie', 'taux_mortalite', 'fertilite_visee', 'cout_aliment', 'surface_m2', 'experience_equipe', 'distance_labo', 'demande_marche', 'saison', 'budget', 'meilleure_souche']

Colonnes labos    : ['id_labo', 'nom_laboratoire', 'type_laboratoire', 'ville', 'region', 'latitude', 'longitude', 'adresse', 'telephone', 'email', 'laborantin_principal', 'nb_laborantins', 'annees_experience_labo', 'specialites_principales', 'maladies_avicoles_traitees', 'nb_analyses_totales', 'nb_analyses_avicoles', 'taux_reussite_pct', 'note_satisfaction', 'cout_analyse_moyen_tnd', 'cout_urgence_tnd', 'delai_standard_jours', 'delai_urgence_heures', 'temps_analyse_moyen_heures', 'capacite_journaliere_analyses', 'charge_actuelle_pct', 'slots_disponibles_semaine', 'delai_pro

<a id='4'></a>
## 4. Nettoyage & Normalisation

In [5]:
BOOL_COLS_ANALYSES = ["effectuee", "conforme"]
BOOL_COLS_LABOS    = [
    "accepte_urgence", "certifie_iso", "agree_ministere_agriculture",
    "equipement_pcr", "equipement_elisa", "equipement_sequencage",
    "ouvert_weekend", "actif"
]

def clean_df(df: pd.DataFrame, bool_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].map(
                lambda x: True if str(x).lower() in ("true","1","oui","yes") else False
            )
    return df.fillna("N/A")

df_analyses = clean_df(df_analyses, BOOL_COLS_ANALYSES)
df_labos    = clean_df(df_labos,    BOOL_COLS_LABOS)

# ── Statistiques rapides ───────────────────────────────────────────────────
print("  Statistiques – Analyses")
print(f"   Total              : {len(df_analyses):,}")
if "conforme" in df_analyses.columns:
    pct_nc = (~df_analyses["conforme"]).sum() / len(df_analyses) * 100
    print(f"   Non conformes      : {pct_nc:.1f}%")
if "meilleure_souche" in df_analyses.columns:
    print(f"   Souches distinctes : {df_analyses['meilleure_souche'].nunique()}")

print("\n  Statistiques – Laboratoires")
print(f"   Total              : {len(df_labos):,}")
print(f"   Actifs             : {df_labos['actif'].sum()}")
print(f"   Acceptent urgences : {df_labos['accepte_urgence'].sum()}")
print(f"   Score global moy.  : {df_labos['score_global'].mean():.1f}")

  Statistiques – Analyses
   Total              : 5,000
   Souches distinctes : 4

  Statistiques – Laboratoires
   Total              : 500
   Actifs             : 450
   Acceptent urgences : 249
   Score global moy.  : 52.1


<a id='5'></a>
## 5. Chunking – Texte structuré par ligne

In [6]:
FIELDS_ANALYSE = [
    ("id_centre", "ID centre"), ("ville", "Ville"), ("region", "Région"),
    ("type_production", "Type de production"), ("meilleure_souche", "Meilleure souche"),
    ("capacite", "Capacité"), ("temperature_moyenne", "Température moyenne"),
    ("humidite", "Humidité"), ("altitude", "Altitude"),
    ("biosecurite_score", "Score biosécurité"), ("historique_maladie", "Historique maladie"),
    ("taux_mortalite", "Taux de mortalité"), ("fertilite_visee", "Fertilité visée"),
    ("cout_aliment", "Coût aliment"), ("surface_m2", "Surface"),
    ("experience_equipe", "Expérience équipe"), ("distance_labo", "Distance au labo"),
    ("demande_marche", "Demande marché"), ("saison", "Saison"), ("budget", "Budget"),
]

FIELDS_LABO = [
    ("id_labo", "ID laboratoire"), ("nom_laboratoire", "Nom du laboratoire"),
    ("type_laboratoire", "Type"), ("ville", "Ville"), ("region", "Région"),
    ("laborantin_principal", "Laborantin principal"), ("nb_laborantins", "Nb laborantins"),
    ("annees_experience_labo", "Années d'expérience"), ("specialites_principales", "Spécialités"),
    ("maladies_avicoles_traitees", "Maladies traitées"), ("nb_analyses_avicoles", "Nb analyses avicoles"),
    ("taux_reussite_pct", "Taux de réussite"), ("note_satisfaction", "Note satisfaction"),
    ("cout_analyse_moyen_tnd", "Coût standard (TND)"), ("cout_urgence_tnd", "Coût urgence (TND)"),
    ("delai_standard_jours", "Délai standard (j)"), ("delai_urgence_heures", "Délai urgence (h)"),
    ("capacite_journaliere_analyses", "Capacité/jour"), ("charge_actuelle_pct", "Charge actuelle"),
    ("slots_disponibles_semaine", "Slots dispo"), ("delai_prochain_rdv_jours", "Prochain RDV (j)"),
    ("accepte_urgence", "Urgences"), ("distance_moyenne_centres_km", "Distance moy. centres (km)"),
    ("certifie_iso", "ISO"), ("agree_ministere_agriculture", "Agréé Min. Agriculture"),
    ("equipement_pcr", "PCR"), ("equipement_elisa", "ELISA"), ("equipement_sequencage", "Séquençage"),
    ("score_global", "Score global"),
]

def row_to_chunk(row: pd.Series, fields: list[tuple]) -> str:
    parts = []
    for field, label in fields:
        val = str(row.get(field, ""))
        if val not in ("", "N/A", "nan"):
            parts.append(f"{label} : {val}.")
    return " ".join(parts)

chunks_analyses, chunks_labos = [], []

for _, row in df_analyses.iterrows():
    chunks_analyses.append({
        "id": str(uuid.uuid4()), "text": row_to_chunk(row, FIELDS_ANALYSE),
        "metadata": {
            "id_centre": str(row.get("id_centre","")), "ville": str(row.get("ville","")),
            "region": str(row.get("region","")), "type_production": str(row.get("type_production","")),
            "meilleure_souche": str(row.get("meilleure_souche","")),
            "biosecurite_score": str(row.get("biosecurite_score","")),
            "taux_mortalite": str(row.get("taux_mortalite","")),
            "conforme": str(row.get("conforme","")),
        }
    })

for _, row in df_labos.iterrows():
    if str(row.get("actif", "False")).lower() != "true":
        continue
    chunks_labos.append({
        "id": str(uuid.uuid4()), "text": row_to_chunk(row, FIELDS_LABO),
        "metadata": {
            "id_labo": str(row.get("id_labo","")), "nom_laboratoire": str(row.get("nom_laboratoire","")),
            "ville": str(row.get("ville","")), "region": str(row.get("region","")),
            "type_laboratoire": str(row.get("type_laboratoire","")),
            "accepte_urgence": str(row.get("accepte_urgence","")),
            "score_global": str(row.get("score_global","")),
            "taux_reussite_pct": str(row.get("taux_reussite_pct","")),
            "delai_standard_jours": str(row.get("delai_standard_jours","")),
            "certifie_iso": str(row.get("certifie_iso","")),
        }
    })

print(f" {len(chunks_analyses):,} chunks analyses générés")
print(f" {len(chunks_labos):,} chunks laboratoires générés (actifs uniquement)")
print("\nExemple chunk analyse :")
print(chunks_analyses[0]["text"][:250] + "...")

 5,000 chunks analyses générés
 450 chunks laboratoires générés (actifs uniquement)

Exemple chunk analyse :
ID centre : 1. Ville : Tunis. Région : Nord. Type de production : Oeuf. Meilleure souche : Lohmann Brown. Capacité : 6012. Température moyenne : 25.5. Humidité : 63. Altitude : 228. Score biosécurité : 85. Historique maladie : Faible. Taux de mortali...


<a id='6'></a>
## 6. Embeddings TF-IDF — Vectorisation rapide

### Pourquoi TF-IDF plutôt que multilingual-e5-base ?

| Critère | `e5-base` (ancienne version) | **TF-IDF (cette version)** |
|---------|----------------------------|--------------------------|
| Taille modèle | ~450 MB | < 1 MB |
| Temps de chargement | 2–5 min | < 1 sec |
| Temps d'encodage (5 000 lignes) | ~3 min | **< 5 sec** |
| GPU requis | recommandé | non |
| Cache disque | non | **oui (joblib)** |
| Qualité sémantique | élevée | bonne (données structurées) |



> Le TF-IDF capture efficacement le vocabulaire du domaine sans nécessiter de représentation vectorielle dense.

In [7]:
import time
from sklearn.metrics.pairwise import cosine_similarity

def build_or_load_tfidf(
    texts: list[str],
    cache_path: str,
    label: str,
    max_features: int = 8000,
) -> tuple:
    """
    Construit le TF-IDF et met en cache les vecteurs + le vectoriseur.
    Au 2e appel, recharge depuis le cache (< 1 sec).
    
    Returns:
        (vectorizer, sparse_matrix)
    """
    vec_path = cache_path + ".vectorizer.pkl"
    mat_path = cache_path + ".matrix.pkl"

    if Path(vec_path).exists() and Path(mat_path).exists():
        t0 = time.time()
        vectorizer = joblib.load(vec_path)
        matrix     = joblib.load(mat_path)
        print(f"{label} : cache chargé en {time.time()-t0:.2f}s – shape {matrix.shape}")
        return vectorizer, matrix

    t0 = time.time()
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),      # unigrammes + bigrammes
        sublinear_tf=True,       # log(tf+1) → atténue les très hautes fréquences
        min_df=2,                # ignore les termes vus < 2 fois
        analyzer="word",
    )
    matrix = vectorizer.fit_transform(texts)
    elapsed = time.time() - t0

    joblib.dump(vectorizer, vec_path)
    joblib.dump(matrix,     mat_path)
    print(f"{label} : {len(texts):,} textes → {matrix.shape} en {elapsed:.2f}s (cache sauvegardé)")
    return vectorizer, matrix


texts_analyses = [c["text"] for c in chunks_analyses]
texts_labos    = [c["text"] for c in chunks_labos]

tfidf_a, matrix_a = build_or_load_tfidf(texts_analyses, TFIDF_A_PATH, "Analyses")
tfidf_l, matrix_l = build_or_load_tfidf(texts_labos,    TFIDF_L_PATH, "Laboratoires")

print("\nDimensions des matrices sparses :")
print(f"   Analyses      : {matrix_a.shape} → {matrix_a.nnz:,} valeurs non-nulles")
print(f"   Laboratoires  : {matrix_l.shape} → {matrix_l.nnz:,} valeurs non-nulles")

Analyses : cache chargé en 0.06s – shape (5000, 8000)
Laboratoires : cache chargé en 0.02s – shape (450, 2238)

Dimensions des matrices sparses :
   Analyses      : (5000, 8000) → 496,993 valeurs non-nulles
   Laboratoires  : (450, 2238) → 73,516 valeurs non-nulles


<a id='7'></a>
## 7. Modèle Random Forest

Deux modèles sont entraînés :

1. **`rf_souche`** – Prédit la meilleure souche pour un centre d'élevage  
   *Features* : biosécurité, mortalité, fertilité visée, type de production, saison, budget…

2. **`rf_labo`** – Classe les laboratoires par pertinence pour une demande d'analyse  
   *Features* : score global, taux de réussite, délai, charge, distance, équipements…

> Modele le plus efficace selon Knime, 
> le Random Forest gère nativement les features mixtes (numériques + catégorielles encodées),
> résiste au surapprentissage sur 5 000 lignes, et produit une **importance des features** exploitable métier.

In [8]:
# ══════════════════════════════════════════════════════════════════
# 7.1 – Random Forest Souche
# ══════════════════════════════════════════════════════════════════

def prepare_features_souche(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Prépare les features numériques et catégorielles pour prédire la souche."""
    num_cols = [
        "capacite", "temperature_moyenne", "humidite", "altitude",
        "biosecurite_score", "taux_mortalite", "fertilite_visee",
        "cout_aliment", "surface_m2", "experience_equipe", "distance_labo", "budget"
    ]
    cat_cols = ["type_production", "saison", "region", "demande_marche"]
    target   = "meilleure_souche"

    available_num = [c for c in num_cols if c in df.columns]
    available_cat = [c for c in cat_cols if c in df.columns]

    X = df[available_num].copy()
    for col in available_num:
        numeric_col = pd.to_numeric(X[col], errors="coerce")
        X[col] = numeric_col.fillna(numeric_col.median())

    for col in available_cat:
        X[col] = LabelEncoder().fit_transform(df[col].astype(str))

    y = df[target].astype(str) if target in df.columns else None
    return X, y


def train_or_load_rf(model_path: str, X, y, label: str, **rf_kwargs) -> RandomForestClassifier:
    """Entraîne le RF ou le recharge depuis le cache."""
    if Path(model_path).exists():
        model = joblib.load(model_path)
        print(f"{label} : modèle chargé depuis cache")
        return model

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier(random_state=42, n_jobs=-1, **rf_kwargs)
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f" {label} entraîné en {elapsed:.2f}s | Accuracy : {acc:.3f}")

    joblib.dump(model, model_path)
    print(f"     Modèle sauvegardé → {model_path}")
    return model


print("  Entraînement Random Forest – Souche")
X_souche, y_souche = prepare_features_souche(df_analyses)
rf_souche = train_or_load_rf(
    RF_SOUCHE_PATH, X_souche, y_souche, "RF Souche",
    n_estimators=200, max_depth=12, min_samples_leaf=3
)

# ── Importance des features ────────────────────────────────────────────────
feat_imp = pd.Series(rf_souche.feature_importances_, index=X_souche.columns)
print("\nTop 8 features – prédiction souche :")
print(feat_imp.nlargest(8).to_string())

  Entraînement Random Forest – Souche
RF Souche : modèle chargé depuis cache

Top 8 features – prédiction souche :
type_production        0.621817
temperature_moyenne    0.086574
biosecurite_score      0.047532
taux_mortalite         0.043740
humidite               0.038631
capacite               0.032050
surface_m2             0.020196
altitude               0.019919


In [9]:
# ══════════════════════════════════════════════════════════════════
# 7.2 – Random Forest Laboratoire
# ══════════════════════════════════════════════════════════════════

def prepare_features_labo(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Features pour classer les labos (target : tier de qualité)."""
    num_cols = [
        "score_global", "taux_reussite_pct", "note_satisfaction",
        "delai_standard_jours", "delai_urgence_heures",
        "capacite_journaliere_analyses", "charge_actuelle_pct",
        "slots_disponibles_semaine", "nb_analyses_avicoles",
        "annees_experience_labo", "cout_analyse_moyen_tnd",
        "distance_moyenne_centres_km",
    ]
    bool_cols = ["accepte_urgence", "certifie_iso", "equipement_pcr", "equipement_elisa"]
    cat_cols  = ["type_laboratoire", "region"]

    available_num  = [c for c in num_cols  if c in df.columns]
    available_bool = [c for c in bool_cols if c in df.columns]
    available_cat  = [c for c in cat_cols  if c in df.columns]

    X = df[available_num].copy()
    for col in available_num:
        X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)
    for col in available_bool:
        X[col] = df[col].map(lambda v: 1 if str(v).lower() == "true" else 0)
    for col in available_cat:
        X[col] = LabelEncoder().fit_transform(df[col].astype(str))

    # Target : tier basé sur score_global (Excellent / Bon / Passable)
    if "score_global" in df.columns:
        scores = pd.to_numeric(df["score_global"], errors="coerce").fillna(0)
        y = pd.cut(scores, bins=[0, 60, 75, 100], labels=["Passable", "Bon", "Excellent"])
    else:
        y = None
    return X, y


df_labos_actifs = df_labos[df_labos["actif"] == True].copy()
print(f"Entraînement Random Forest – Laboratoire ({len(df_labos_actifs)} labos actifs)")
X_labo, y_labo = prepare_features_labo(df_labos_actifs)
rf_labo = train_or_load_rf(
    RF_LABO_PATH, X_labo, y_labo.astype(str), "RF Labo",
    n_estimators=150, max_depth=10, min_samples_leaf=5
)

feat_imp_l = pd.Series(rf_labo.feature_importances_, index=X_labo.columns)
print("\nTop 8 features – classification labo :")
print(feat_imp_l.nlargest(8).to_string())

Entraînement Random Forest – Laboratoire (450 labos actifs)
RF Labo : modèle chargé depuis cache

Top 8 features – classification labo :
score_global                     0.730439
charge_actuelle_pct              0.046878
taux_reussite_pct                0.038091
note_satisfaction                0.030512
distance_moyenne_centres_km      0.029340
slots_disponibles_semaine        0.021385
capacite_journaliere_analyses    0.018560
nb_analyses_avicoles             0.017712


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 7.3 – Fonctions de prédiction (utilisées dans le pipeline RAG)
# ══════════════════════════════════════════════════════════════════

def predict_souche(centre_features: dict) -> dict:
    """
    Prédit la meilleure souche pour un centre donné.
    
    Args:
        centre_features : dict avec les caractéristiques du centre
    Returns:
        {'souche': str, 'confiance': float, 'alternatives': list}
    """
    X, _ = prepare_features_souche(pd.DataFrame([centre_features]))
    # Aligner les colonnes avec celles du modèle
    for col in rf_souche.feature_names_in_:
        if col not in X.columns:
            X[col] = 0
    X = X[rf_souche.feature_names_in_]
    
    proba = rf_souche.predict_proba(X)[0]
    classes = rf_souche.classes_
    top3_idx = np.argsort(proba)[::-1][:3]
    
    return {
        "souche"       : classes[top3_idx[0]],
        "confiance"    : round(float(proba[top3_idx[0]]) * 100, 1),
        "alternatives" : [(classes[i], round(float(proba[i])*100, 1)) for i in top3_idx[1:]]
    }


def score_labos_rf(df_labos_subset: pd.DataFrame) -> pd.DataFrame:
    """
    Applique le RF Labo sur un sous-ensemble pour obtenir le tier de chaque labo.
    Returns: df avec colonnes 'tier_rf' et 'proba_excellent'.
    """
    X, _ = prepare_features_labo(df_labos_subset)
    for col in rf_labo.feature_names_in_:
        if col not in X.columns:
            X[col] = 0
    X = X[rf_labo.feature_names_in_]
    
    preds = rf_labo.predict(X)
    proba = rf_labo.predict_proba(X)
    excellent_idx = list(rf_labo.classes_).index("Excellent") if "Excellent" in rf_labo.classes_ else 0
    
    result = df_labos_subset.copy()
    result["tier_rf"]        = preds
    result["proba_excellent"] = proba[:, excellent_idx]
    return result.sort_values("proba_excellent", ascending=False)


print(" Fonctions RF prêtes : predict_souche() | score_labos_rf()")

# ── Test rapide ────────────────────────────────────────────────────────────
demo_centre = {
    "type_production": "Poulet de chair", "saison": "Été", "region": "Sfax",
    "biosecurite_score": 7.5, "taux_mortalite": 3.2, "fertilite_visee": 85,
    "temperature_moyenne": 28, "humidite": 55, "budget": 50000
}
try:
    pred = predict_souche(demo_centre)
    print(f"\n Prédiction souche (demo) :")
    print(f"   Recommandée  : {pred['souche']} ({pred['confiance']}% de confiance)")
    print(f"   Alternatives : {pred['alternatives']}")
except Exception as e:
    print(f" Demo ignorée (colonnes absentes du jeu réel) : {e}")

 Fonctions RF prêtes : predict_souche() | score_labos_rf()

 Prédiction souche (demo) :
   Recommandée  : Hybrid Converter (67.8% de confiance)
   Alternatives : [('Lohmann Brown', 23.3), ('Ross 308', 5.1)]


<a id='8'></a>
## 8. Indexation ChromaDB

Les vecteurs TF-IDF **denses** (`.toarray()`) sont stockés dans ChromaDB pour le retrieval cosine.

> **Note** : ChromaDB travaille sur des vecteurs denses. La conversion sparse→dense est faite ligne par ligne pour économiser la RAM.

In [11]:
client = chromadb.PersistentClient(path=CHROMA_PATH)

def index_collection_tfidf(client, name: str, chunks: list[dict], matrix) -> chromadb.Collection:
    """
    Crée (ou recrée) une collection ChromaDB avec les embeddings TF-IDF.
    """
    try:
        client.delete_collection(name=name)
    except Exception:
        pass
    col = client.get_or_create_collection(name=name, metadata={"hnsw:space": "cosine"})

    # Ajout par batchs de 512 pour éviter les timeouts
    BATCH = 512
    for start in tqdm(range(0, len(chunks), BATCH), desc=f"Indexation {name}"):
        end = min(start + BATCH, len(chunks))
        batch_chunks = chunks[start:end]
        batch_vecs   = matrix[start:end].toarray().tolist()
        col.add(
            ids        =[c["id"] for c in batch_chunks],
            embeddings  =batch_vecs,
            metadatas   =[c["metadata"] for c in batch_chunks],
            documents   =[c["text"] for c in batch_chunks],
        )
    print(f" Collection '{name}' : {col.count():,} documents indexés")
    return col


col_analyses = index_collection_tfidf(client, COLLECTION_ANALYSES, chunks_analyses, matrix_a)
col_labos    = index_collection_tfidf(client, COLLECTION_LABOS,    chunks_labos,    matrix_l)

Indexation poulina_analyses: 100%|██████████| 10/10 [00:23<00:00,  2.34s/it]


 Collection 'poulina_analyses' : 5,000 documents indexés


Indexation poulina_laboratoires: 100%|██████████| 1/1 [00:01<00:00,  1.11s/it]

 Collection 'poulina_laboratoires' : 450 documents indexés


<a id='9'></a>
## 9. Retrieval & Routage

In [12]:
# ── Poids pour le routage ─────────────────────────────────────────────────
WEIGHTS_LABO = {
    "laboratoire": 2.0, "labo": 2.0, "laborantin": 1.5,
    "analyse": 1.5, "analyser": 1.5, "urgent": 1.5, "urgence": 1.5,
    "délai": 1.0, "disponible": 1.0, "distance": 1.0,
    "proche": 1.0, "compétent": 1.0, "certifié": 1.0,
    "pcr": 1.0, "elisa": 1.0,
}
WEIGHTS_SOUCHE = {
    "souche": 2.0, "élevage": 1.5, "centre": 1.5, "bâtiment": 1.0,
    "fertilité": 1.0, "mortalité": 1.0, "maladie": 1.5,
    "salmonelle": 1.5, "newcastle": 1.5, "biosécurité": 1.0,
    "conforme": 1.0, "production": 1.0, "poulet": 1.0, "dinde": 1.0,
}

def route_question(question: str) -> str:
    q = question.lower()
    score_labo   = sum(w for kw, w in WEIGHTS_LABO.items()   if kw in q)
    score_souche = sum(w for kw, w in WEIGHTS_SOUCHE.items() if kw in q)
    if score_labo > 0 and score_souche > 0:
        return "both"
    return "labos" if score_labo >= score_souche else "analyses"


def retrieve_from_tfidf(
    collection,
    vectorizer: TfidfVectorizer,
    question: str,
    top_k: int = TOP_K,
    score_threshold: float = SCORE_THRESH,
    where_filter: dict | None = None,
) -> list[dict]:
    """Encode la question avec le TF-IDF et interroge ChromaDB."""
    q_vec = vectorizer.transform([question]).toarray()[0].tolist()
    results = collection.query(
        query_embeddings=[q_vec],
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances"],
    )
    retrieved = []
    if results["ids"] and results["ids"][0]:
        for i, dist in enumerate(results["distances"][0]):
            score = 1 - dist if dist is not None else 0.0
            if score >= score_threshold:
                retrieved.append({
                    "score"   : round(score, 4),
                    "text"    : results["documents"][0][i],
                    "metadata": results["metadatas"][0][i] or {},
                })
    return retrieved


def retrieve(
    question: str,
    top_k: int = TOP_K,
    filtre_centre: str | None = None,
    filtre_labo: str | None = None,
    filtre_ville: str | None = None,
    force_collection: str | None = None,
) -> tuple[list[dict], list[dict]]:
    target = force_collection or route_question(question)

    where_a = ({"id_centre": filtre_centre} if filtre_centre else None)
    where_l = {}
    if filtre_labo:  where_l["id_labo"] = filtre_labo
    if filtre_ville: where_l["ville"]   = filtre_ville
    where_l = where_l or None

    chunks_a = retrieve_from_tfidf(col_analyses, tfidf_a, question, top_k, where_filter=where_a) \
               if target in ("analyses", "both") else []
    chunks_l = retrieve_from_tfidf(col_labos,    tfidf_l, question, top_k, where_filter=where_l) \
               if target in ("labos", "both") else []
    return chunks_a, chunks_l


# ── Test de routage ────────────────────────────────────────────────────────
tests = [
    "Quel laboratoire proche de Tunis accepte les urgences ?",
    "Quelle souche recommandes-tu pour un élevage de poulet ?",
    "Mon élevage a un historique de Newcastle – quel labo et quelle souche ?",
]
print("Tests de routage :")
for q in tests:
    print(f"   [{route_question(q):8}] {q}")

Tests de routage :
   [labos   ] Quel laboratoire proche de Tunis accepte les urgences ?
   [analyses] Quelle souche recommandes-tu pour un élevage de poulet ?
   [both    ] Mon élevage a un historique de Newcastle – quel labo et quelle souche ?


<a id='10'></a>
## 10. Pipeline RAG complet (TF-IDF + RF + Mistral)

```
Question
   │
   ├─► Routage (keyword scoring)
   │
   ├─► TF-IDF Retrieval ──────► ChromaDB (analyses + labos)
   │
   ├─► Random Forest ──────────► Prédiction souche / scoring labo
   │
   └─► Mistral (contexte RAG + prédiction RF) ──► Réponse finale
```

In [25]:
SYSTEM_PROMPT = """
Tu es l'assistant IA de Poulina, expert en élevage de volailles en Tunisie.
Tu analyses les données des centres d'élevage ET des laboratoires partenaires
pour aider les responsables à prendre les meilleures décisions.

## Tes capacités
- Identifier la meilleure souche pour chaque centre d'élevage
- Détecter et prévenir les maladies critiques (Salmonelle, Newcastle, Mycoplasme…)
- Recommander le meilleur laboratoire selon : distance, disponibilité, compétences, coût, urgence
- Évaluer la conformité des analyses et signaler les anomalies
- Estimer la fréquence d'analyse recommandée selon la situation sanitaire

## Pour les recommandations de laboratoire, tu tiens compte de :
1. Distance aux centres concernés
2. Disponibilité immédiate (slots, délai prochain RDV)
3. Urgence : délai urgence et acceptation
4. Compétences : spécialités, équipements PCR/ELISA, maladies traitées
5. Fiabilité : taux de réussite, satisfaction, expérience
6. Tier Random Forest (Excellent / Bon / Passable)
7. Coût en TND

## Règles
- Base-toi UNIQUEMENT sur le contexte fourni.
- Si l'information est absente, dis-le clairement.
- Hors sujet : réponds exactement "Je ne peux pas répondre à cette question car elle est hors de mon domaine."
- Sois concis et actionnable. Réponds en français.
""".strip()


def build_context(chunks_a: list[dict], chunks_l: list[dict], rf_souche_pred: dict | None = None) -> str:
    parts = []
    if rf_souche_pred:
        parts.append("=== PRÉDICTION RANDOM FOREST – SOUCHE ===")
        parts.append(f"Souche recommandée : {rf_souche_pred['souche']} (confiance : {rf_souche_pred['confiance']}%)")
        alts = ", ".join(f"{s} ({p}%)" for s, p in rf_souche_pred["alternatives"])
        parts.append(f"Alternatives       : {alts}")
        parts.append("")
    if chunks_a:
        parts.append("=== DONNÉES ÉLEVAGE / SOUCHES ===")
        for i, r in enumerate(chunks_a, 1):
            parts.append(f"[Élevage {i} – score {r['score']}]")
            parts.append(r["text"])
            parts.append("")
    if chunks_l:
        parts.append("=== DONNÉES LABORATOIRES ===")
        for i, r in enumerate(chunks_l, 1):
            tier = r["metadata"].get("tier_rf", "")
            tier_str = f" | Tier RF : {tier}" if tier else ""
            parts.append(f"[Laboratoire {i} – score {r['score']}{tier_str}]")
            parts.append(r["text"])
            parts.append("")
    return "\n".join(parts) if parts else "Aucune donnée pertinente trouvée."


def ask_poulina(
    question: str,
    filtre_centre: str | None = None,
    filtre_labo: str | None = None,
    filtre_ville: str | None = None,
    force_collection: str | None = None,
    centre_features: dict | None = None,   # ← pour activer la prédiction RF souche
    verbose: bool = True,
) -> str:
    """
    Pipeline RAG complet : question → routage → TF-IDF retrieval → RF → Mistral.

    Args:
        question         : Question en langage naturel
        filtre_centre    : Restreindre aux analyses d'un centre précis (id)
        filtre_labo      : Restreindre aux données d'un labo précis (id)
        filtre_ville     : Restreindre les labos à une ville précise
        force_collection : 'analyses', 'labos' ou 'both'
        centre_features  : Dict de caractéristiques du centre → active predict_souche()
        verbose          : Afficher le détail du retrieval
    Returns:
        Réponse textuelle de Mistral
    """
    # 1. Retrieval TF-IDF
    chunks_a, chunks_l = retrieve(
        question, filtre_centre=filtre_centre, filtre_labo=filtre_labo,
        filtre_ville=filtre_ville, force_collection=force_collection,
    )

    # 2. Prédiction RF souche (optionnel)
    rf_pred = None
    if centre_features:
        try:
            rf_pred = predict_souche(centre_features)
        except Exception as e:
            print(f" RF souche ignoré : {e}")

    if verbose:
        target = force_collection or route_question(question)
        print(f"  Routage        : {target}")
        print(f" Chunks         : {len(chunks_a)} élevage | {len(chunks_l)} labo")
        if rf_pred:
            print(f"RF Souche      : {rf_pred['souche']} ({rf_pred['confiance']}%)")
        for r in chunks_a:
            print(f"   [analyse {r['score']}] centre={r['metadata'].get('id_centre','?')} "
                  f"souche={r['metadata'].get('meilleure_souche','?')} "
                  f"conforme={r['metadata'].get('conforme','?')}")
        for r in chunks_l:
            print(f"   [labo    {r['score']}] {r['metadata'].get('nom_laboratoire','?')} | "
                  f"{r['metadata'].get('ville','?')} | "
                  f"urgence={r['metadata'].get('accepte_urgence','?')} | "
                  f"score={r['metadata'].get('score_global','?')}")
        print()

    # 3. Construction du contexte
    context = build_context(chunks_a, chunks_l, rf_souche_pred=rf_pred)
    user_msg = f"Contexte (données Poulina) :\n{context}\n\n---\nQuestion : {question}"

    # 4. Appel Mistral
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_msg),
    ])
    return response.content


print(" Pipeline ask_poulina() prêt (TF-IDF + RF + Mistral)")

 Pipeline ask_poulina() prêt (TF-IDF + RF + Mistral)


<a id='11'></a>
## 11. Tests métier – Scénarios Poulina

Les tests couvrent les **6 cas d'usage principaux** du cahier des charges.

In [14]:
# ── Helpers d'affichage ────────────────────────────────────────────────────
def section(title: str, num: int):
    print("\n" + "═" * 65)
    print(f"  TEST {num} │ {title}")
    print("═" * 65)

In [15]:
# ── Test 1 : Non-conformités ──────────────────────────────────────────────
section("Détection des non-conformités", 1)
print(ask_poulina("Quels centres d'élevage ont des analyses non conformes ?"))


═════════════════════════════════════════════════════════════════
  TEST 1 │ Détection des non-conformités
═════════════════════════════════════════════════════════════════
  Routage        : both
 Chunks         : 0 élevage | 0 labo

Aucune donnée pertinente n'est disponible dans le contexte fourni pour identifier les centres d'élevage avec des analyses non conformes. Je ne peux donc pas répondre à cette question.


In [16]:
# ── Test 2 : Meilleure souche avec Random Forest ──────────────────────────
section("Recommandation souche (RAG + Random Forest)", 2)
rep = ask_poulina(
    "Quelle est la meilleure souche pour un élevage de poulet à Sfax en été avec un score biosécurité de 8 ?",
    centre_features={
        "type_production": "Poulet de chair", "saison": "Été", "region": "Sfax",
        "biosecurite_score": 8.0, "taux_mortalite": 2.5, "fertilite_visee": 90,
        "temperature_moyenne": 30, "humidite": 50, "budget": 60000,
    },
    verbose=True,
)
print(rep)


═════════════════════════════════════════════════════════════════
  TEST 2 │ Recommandation souche (RAG + Random Forest)
═════════════════════════════════════════════════════════════════
  Routage        : analyses
 Chunks         : 5 élevage | 0 labo
🌲 RF Souche      : Hybrid Converter (68.5%)
   [analyse 0.2291] centre=3157 souche=Cobb 500 conforme=
   [analyse 0.2202] centre=1148 souche=Cobb 500 conforme=
   [analyse 0.2086] centre=4320 souche=Cobb 500 conforme=
   [analyse 0.2078] centre=3454 souche=Ross 308 conforme=
   [analyse 0.2069] centre=3141 souche=Ross 308 conforme=

**Réponse :**
D'après le contexte fourni, **aucun des élevages listés n'a un score de biosécurité de 8** (les scores varient entre 63 et 98). Cependant, pour un élevage de poulet à Sfax en été avec un **score de biosécurité faible (ex. 8/100)**, voici les recommandations basées sur les données disponibles :

### **Souche recommandée : Cobb 500**
- **Pourquoi ?**
  - **Adaptée aux conditions locales** : Tous l

In [17]:
# ── Test 3 : Alerte Salmonelle ────────────────────────────────────────────
section("Alerte maladie critique – Salmonelle", 3)
print(ask_poulina(
    "Y a-t-il des détections de Salmonelle ? Quels centres sont à risque ? "
    "Quel laboratoire contacter en urgence ?",
    force_collection="both", verbose=True,
))


═════════════════════════════════════════════════════════════════
  TEST 3 │ Alerte maladie critique – Salmonelle
═════════════════════════════════════════════════════════════════
  Routage        : both
 Chunks         : 0 élevage | 5 labo
   [labo    0.1143] Laboratoire Central de Tunis | Mahdia | urgence=False | score=69.67
   [labo    0.1114] Laboratoire Régional Sousse | Jendouba | urgence=False | score=52.41
   [labo    0.1103] Laboratoire Central de Tunis | Gabès | urgence=False | score=53.27
   [labo    0.1098] Laboratoire Central de Tunis | Jendouba | urgence=False | score=40.4
   [labo    0.1095] Laboratoire Central de Tunis | Béja | urgence=True | score=47.32

### **1. Détection de Salmonelle**
D'après les données fournies, **aucune information explicite sur des détections récentes de Salmonelle** dans les centres d'élevage n'est disponible. Une analyse immédiate est recommandée pour confirmer ou infirmer sa présence.

---

### **2. Centres à risque (priorisation)**
Bien qu

In [21]:
# ── Test 4 : Meilleur labo (filtre ville) ─────────────────────────────────
section("Recommandation laboratoire urgent – Sfax", 4)
print(ask_poulina(
    "Quel laboratoire disponible accepte les urgences et est le plus compétent "
    "pour analyser la Salmonelle ?",
    filtre_ville="Sfax", verbose=True,
))


═════════════════════════════════════════════════════════════════
  TEST 4 │ Recommandation laboratoire urgent – Sfax
═════════════════════════════════════════════════════════════════
  Routage        : both
 Chunks         : 0 élevage | 2 labo
   [labo    0.1018] Laboratoire Régional Sousse | Sfax | urgence=True | score=51.65
   [labo    0.1013] Institut Vétérinaire Gabès | Sfax | urgence=False | score=49.0

D'après les données fournies, voici la réponse :

**Laboratoire recommandé :**
**Laboratoire Régional Sousse (LAB0209)**
- **Accepte les urgences** : Oui (délai urgence : 27h)
- **Compétent pour la Salmonelle** : Oui (maladie traitée, avec PCR/ELISA disponibles)
- **Score global** : 51,65 (meilleur que l'autre laboratoire)
- **Taux de réussite** : 93,3% (supérieur à 81,2%)
- **Note satisfaction** : 4,2/5
- **Coût urgence** : 191 TND
- **Slots disponibles** : 28 (meilleure capacité que LAB0280)

**Alternative non urgente :**
L’**Institut Vétérinaire Gabès (LAB0280)** traite aussi 

In [22]:
# ── Test 5 : Question duale (souche + labo) ──────────────────────────────
section("Question duale – Souche + Laboratoire (Newcastle)", 5)
print(ask_poulina(
    "Mon élevage a un historique de maladie Newcastle. Quelle souche recommandes-tu "
    "et quel laboratoire agréé peut analyser en moins de 48h ?",
    force_collection="both", verbose=True,
))


═════════════════════════════════════════════════════════════════
  TEST 5 │ Question duale – Souche + Laboratoire (Newcastle)
═════════════════════════════════════════════════════════════════
  Routage        : both
 Chunks         : 1 élevage | 0 labo
   [analyse 0.1052] centre=1819 souche=Lohmann Brown conforme=

### **Recommandation pour l'Élevage 1 (Tozeur) – Newcastle**

#### **1. Souche recommandée**
**Lohmann Brown** reste la meilleure option pour votre élevage (déjà identifiée comme optimale dans le contexte).
**Pourquoi ?**
- Bonne résistance historique aux pathogènes (score biosécurité élevé : 81).
- Adaptée à la production d’œufs en climat chaud (température moyenne : 29,6°C).
- **Précautions spécifiques contre Newcastle** :
  - Vaccination renforcée (souches *LaSota* ou *I-2* selon protocole vétérinaire).
  - Surveillance accrue des signes cliniques (toux, torticolis, chute de ponte).
  - Renforcer la biosécurité (quarantaine, désinfection des véhicules).

---

#### **2. 

In [23]:
# ── Test 6 : Hors sujet ──────────────────────────────────────────────────
section("Robustesse – Question hors domaine", 6)
print(ask_poulina("Quelle est la météo à Tunis demain ?", verbose=False))


═════════════════════════════════════════════════════════════════
  TEST 6 │ Robustesse – Question hors domaine
═════════════════════════════════════════════════════════════════
Je ne peux pas répondre à cette question car elle est hors de mon domaine.


# - Récapitulatif des améliorations v2.0

| Composant | v1 (originale) | v2 (cette version) |
|-----------|----------------|-------------------|
| Embeddings | multilingual-e5-base (~3 min) |  TF-IDF sparse (&lt; 5 sec + cache) |
| Prédiction souche | RAG seul (Mistral) |  Random Forest + RAG |
| Scoring labo | Règles manuelles |  Random Forest (tier Excellent/Bon/Passable) |
| Cache disque | Non |  joblib (vecteurs + modèles RF) |
| GPU requis | Recommandé |  Non (CPU only) |